In [ ]:
# Unsloth quick install for Kaggle (handles deps)
!pip -q install --upgrade pip
!pip -q install 'unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git'
!pip -q install --upgrade transformers accelerate bitsandbytes

In [ ]:
import os, subprocess, sys
REPO = '/kaggle/working/lawforge'
if not os.path.isdir(REPO):
    subprocess.check_call(['git','clone','--depth','1','https://github.com/PAMF2/lawforge.git', REPO])
sys.path.insert(0, REPO)

In [ ]:
import torch
from unsloth import FastLanguageModel
MODEL = os.environ.get('LAWFORGE_LLM_MODEL', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit')
model, tok = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=2048, dtype=None, load_in_4bit=True)
FastLanguageModel.for_inference(model)
print(f'loaded {MODEL} mem={torch.cuda.memory_allocated()/1e9:.2f}GB')

In [ ]:
import json
from pathlib import Path
INPUTS = Path(f'{REPO}/kaggle/llm_classify_v3/inputs')
rows = []
for s in ['hard2_test','hard3_test']:
    for line in open(INPUTS/f'{s}.jsonl'):
        r = json.loads(line); r['_split']=s; rows.append(r)
print(f'rows: {len(rows)}')

In [ ]:
import time, json, re
from pathlib import Path

SYSTEM = ('You are an expert algebraist. In magma theory (a set G with one binary operation, '
          'written *), decide whether a universally-quantified hypothesis h logically implies '
          'a universally-quantified goal g for ALL magmas. The hypothesis is strong: it must '
          'force the goal in every possible magma, including infinite ones. If even one magma '
          'satisfies h but violates g, the answer is FALSE. Reason briefly (counter-example '
          'attempt + algebraic derivation), then output exactly: ANSWER: TRUE or ANSWER: FALSE')

@torch.inference_mode()
def classify(h, g):
    user = (f'h: forall x y z w u in G, {h}\n'
            f'g: forall x y z w u in G, {g}\n\n'
            f'Does h imply g for all magmas?')
    msgs = [{'role':'system','content':SYSTEM}, {'role':'user','content':user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False,
        pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
    txt = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    upp = txt.upper()
    # find LAST ANSWER: TRUE/FALSE
    m = re.findall(r'ANSWER:\s*(TRUE|FALSE)', upp)
    if m: return m[-1].lower(), txt
    last = txt.strip().split('\n')[-1].upper()
    if 'TRUE' in last and 'FALSE' not in last: return 'true', txt
    if 'FALSE' in last and 'TRUE' not in last: return 'false', txt
    return 'unknown', txt

OUT = Path('/kaggle/working/llm_preds_v3.jsonl')
t0 = time.time(); correct = 0
stats = {'tp_t':0,'fp_t':0,'tp_f':0,'fp_f':0,'unk':0}
with OUT.open('w') as f:
    for i, r in enumerate(rows):
        pred, raw = classify(r['hypothesis'], r['goal'])
        label = r['label']
        if pred == 'true':
            stats['tp_t' if label=='true' else 'fp_t'] += 1
        elif pred == 'false':
            stats['tp_f' if label=='false' else 'fp_f'] += 1
        else: stats['unk'] += 1
        if pred == label: correct += 1
        f.write(json.dumps({'id':r['id'],'split':r['_split'],'label':label,'pred':pred,'raw':raw[-400:]})+'\n')
        f.flush()
        if (i+1) % 20 == 0:
            print(f'[{i+1}/{len(rows)}] correct={correct} stats={stats} t={time.time()-t0:.0f}s', flush=True)
print(f'=== FINAL ===', flush=True)
print(f'accuracy={correct}/{len(rows)} = {correct/len(rows)*100:.1f}%', flush=True)
print(f'stats={stats} time={time.time()-t0:.0f}s', flush=True)